# Roadwork modeling

This notebook runs a baseline model on `data/derived/street_weather_lagged_model.csv`.


In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from pathlib import Path

DERIVED_DIR = Path("data/derived")


In [2]:
df = pd.read_csv(DERIVED_DIR / "street_weather_lagged_model.csv")
summary = {
    "rows": len(df),
    "years": (int(df["year"].min()), int(df["year"].max())),
    "positive_rate": float(df["roadwork_done"].mean()),
    "feature_count": df.shape[1] - 3,
}
summary


{'rows': 276000,
 'years': (2003, 2025),
 'positive_rate': 0.06987318840579711,
 'feature_count': 32}

Before fitting the model, it helps to look at the target itself.

`roadwork_done` is the label being predicted here. A value of 1 means a street got roadwork in that target summer season, and 0 means it did not. This is an imbalanced problem, so raw accuracy by itself can be misleading.

In [ ]:
import matplotlib.pyplot as plt

year_rate = df.groupby("year")["roadwork_done"].mean().reset_index()
class_counts = df["roadwork_done"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["no roadwork", "roadwork"], class_counts.values, color=["#4C78A8", "#E45756"])
axes[0].set_title("Class balance in the modeling table")
axes[0].set_ylabel("Rows")

axes[1].plot(year_rate["year"], year_rate["roadwork_done"], marker="o", color="#F58518")
axes[1].set_title("Roadwork rate by target year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Share of positive rows")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [3]:
predictors = [c for c in df.columns if c not in ["normalized_street_name", "year", "roadwork_done"]]
X = df[predictors].replace([float("inf"), float("-inf")], pd.NA).fillna(0)
y = df["roadwork_done"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]


In [4]:
metrics = {
    "roc_auc": roc_auc_score(y_test, y_prob),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
}
metrics

report = pd.DataFrame(classification_report(y_test, y_pred, digits=4, output_dict=True)).T
report


,precision,recall,f1-score,support
0,0.941715,0.791755,0.860249,77015.000000
1,0.111419,0.347623,0.168751,5785.000000
accuracy,0.760725,0.760725,0.760725,0.760725
macro avg,0.526567,0.569689,0.514500,82800.000000
weighted avg,0.883705,0.760725,0.811936,82800.000000


What ROC-AUC means here:

ROC-AUC measures how well the model ranks positive cases above negative cases across all possible classification thresholds. A value of 0.5 is basically random ranking, and a value of 1.0 is perfect ranking.

That is useful here because the table is imbalanced. Accuracy can look decent even when the model mostly predicts the majority class. ROC-AUC tells you more about whether the model is actually separating the two classes. It still does not replace precision and recall, which matter because positive roadwork cases are relatively rare.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fpr, tpr, color="#4C78A8", label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("ROC curve")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend()

im = axes[1].imshow(cm, cmap="Blues")
axes[1].set_title("Confusion matrix")
axes[1].set_xticks([0, 1], labels=["pred 0", "pred 1"])
axes[1].set_yticks([0, 1], labels=["true 0", "true 1"])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, cm[i, j], ha="center", va="center", color="black")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [5]:
feature_importance = (
    pd.Series(clf.feature_importances_, index=predictors)
    .sort_values(ascending=False)
    .head(15)
    .rename("importance")
)
feature_importance


roadwork_factor_summer_lag1        0.368978
roadwork_factor_summer_lag2        0.309788
temperature_curwinter              0.020839
wind_speed_winter_lag1             0.020336
wind_speed_summer_lag1             0.019496
wind_speed_summer_lag2             0.016729
temperature_summer_lag2            0.014921
wind_speed_curwinter               0.014700
wind_speed_winter_lag2             0.014615
temperature_summer_lag1            0.014146
temperature_winter_lag1            0.013740
temperature_winter_lag2            0.013191
precipitation_summer_lag1          0.012946
precipitation_hours_winter_lag1    0.012941
rain_summer_lag1                   0.011172
Name: importance, dtype: float64

In [ ]:
import matplotlib.pyplot as plt

feature_plot = feature_importance.sort_values()

plt.figure(figsize=(8, 5))
plt.barh(feature_plot.index, feature_plot.values, color="#54A24B")
plt.title("Top feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

Expected result from the current table: prior summer roadwork features matter most, and weather variables matter less.
